# Laboratorio 10: Vectorización, Indexación y Reconocimiento de Rostros
## Base de Datos II - Profesor Heider Sanchez
### P1: Generar los vectores característicos

In [ ]:
import os
import glob

import face_recognition
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import psycopg2

DATASET_PATH = "/home/daros/academico/2026-01/DB2/BD2_LABS/lab11/archive/lfw-funneled"


In [ ]:

coleccion = []

rutas = glob.iglob(os.path.join(DATASET_PATH, "**", "*.jpg"), recursive=True)

for path in rutas:
    person = path.split("/")[-2]
    coleccion.append({"person": person, "path": path})

coleccion = pd.DataFrame(coleccion)

if coleccion.empty:
    print("¡Error! La tabla está vacía. Python no encontró ningún .jpg en la ruta:", DATASET_PATH)
else:
    print(f"¡Éxito! Se encontraron {len(coleccion)} fotos.")
    print(coleccion.head())


In [ ]:

def mostrarFotos(coleccion, posiciones):
    plt.figure(figsize=(16,10))
    i=0
    for idx in posiciones:
        img = plt.imread(coleccion.path.iloc[idx])
        plt.subplot(4, 4, i+1)
        plt.imshow(img)
        plt.title(coleccion.person.iloc[idx]+str(img.shape))
        plt.xticks([])
        plt.yticks([])
        i+=1
    plt.tight_layout()
    plt.show()

In [ ]:

#_________________________ Desarrollo de la funcion embeddings
def generate_face_embeddings(colecion_df, N):
    print("______INICIO_______")
    conn = psycopg2.connect(
        dbname="postgres",
        user="postgres",
        password="123456",
        host="localhost",
        port="5433"
    )
    cur = conn.cursor()
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")  # Habilitar la extensión pgvector [cite: 306, 307]
    cur.execute("""
                CREATE TABLE IF NOT EXISTS face_embeddings
                (   id SERIAL PRIMARY KEY,
                    name TEXT,
                    path TEXT,
                    embedding VECTOR(128));
                """)  # Crear una tabla para almacenar los embeddings faciales de 128 dimensiones [cite: 308, 309, 310, 311, 312, 313]
    conn.commit()

    rostros_procesados = 0

    for index, row in colecion_df.iterrows():
        # guardadmos los rostros procesados
        if rostros_procesados > N:
            break
        nombre = row["person"]
        ruta = row["path"]

        try:
            image = face_recognition.load_image_file(ruta)
            # detectamos los rostros
            face_encodings = face_recognition.face_encodings(image)
            if face_encodings:
                # Tomamos el primer rostro detectado
                vector_128 = face_encodings[0]
                cur.execute(
                    "INSERT INTO face_embeddings (name, path, embedding) VALUES (%s, %s, %s)",
                    (nombre, ruta, vector_128.tolist())
                )
                conn.commit()
                rostros_procesados += 1
                print(f"Rostro {rostros_procesados}/{N} guardado: {nombre}")
            else:
                print(f"No se detectó ningún rostro en la imagen: {ruta}")
        except Exception as e:
            print(f"Error: {e}")
    cur.close()
    conn.close()
    print("¡Proceso finalizado con éxito!")

#generate_face_embeddings(coleccion,100)


#posiciones = list(range(0, 16))
#mostrarFotos(coleccion, posiciones)